In [1]:
import os

In [2]:
pwd

'd:\\ML Projects\\Chest Cancer Classification\\chest-cancer-classifier\\research'

In [3]:
os.chdir("../")

In [4]:
pwd

'd:\\ML Projects\\Chest Cancer Classification\\chest-cancer-classifier'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    params_learning_rate: int

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, 'Chest-CT-Scan-data')
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE,
            params_learning_rate= params.LEARNING_RATE
        )

        return training_config
    

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import json

In [9]:
from pathlib import Path
import tensorflow as tf
import numpy as np

class Training:
    def __init__(self, config):
        self.config = config
        self.model = None
        self.train_generator = None
        self.valid_generator = None

    def get_base_model(self):
        # Clear any existing TensorFlow/Keras session
        tf.keras.backend.clear_session()

        # Load the model
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)

        # Compile the model with a fresh optimizer
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate= self.config.params_learning_rate),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

    def train_valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1. / 255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear",
            class_mode="binary" 
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        print("Valid Dir ", self.config.training_data)
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        print("Train Dir ", self.config.training_data)
        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

        print(self.train_generator.class_indices)

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        if self.model is None:
            raise ValueError("Model not loaded. Please load the model using `get_base_model()` before training.")

        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        # Train the model
        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        # Save the trained model
        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

        save_classes(self.train_generator)
    

In [10]:
def save_classes(train_generator):
    save_dir = "model_with_classes"
    os.makedirs(save_dir, exist_ok=True) 
    with open("model_with_classes/class_indices.json", "w") as f:
        json.dump(train_generator.class_indices, f)

In [11]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2025-08-13 15:36:00,502: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-08-13 15:36:00,509: INFO: common: yaml file: params.yaml loaded successfully]
[2025-08-13 15:36:00,512: INFO: common: created directory at: artifacts]
[2025-08-13 15:36:00,514: INFO: common: created directory at: artifacts\training]
[2025-08-13 15:36:01,607: WARNING: module_wrapper: From c:\Users\Dhaval\miniconda3\envs\cancer\Lib\site-packages\keras\src\backend\common\global_state.py:82: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.
]
Valid Dir  artifacts\data_ingestion\Chest-CT-Scan-data
Found 68 images belonging to 2 classes.
Train Dir  artifacts\data_ingestion\Chest-CT-Scan-data
Found 275 images belonging to 2 classes.
{'adenocarcinoma': 0, 'normal': 1}
Epoch 1/5


c:\Users\Dhaval\miniconda3\envs\cancer\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


17/17 ━━━━━━━━━━━━━━━━━━━━ 55s 3s/step - accuracy: 0.7584 - loss: 0.6396 - val_accuracy: 0.9062 - val_loss: 0.1493
Epoch 2/5
 1/17 ━━━━━━━━━━━━━━━━━━━━ 33s 2s/step - accuracy: 0.8750 - loss: 0.3143

c:\Users\Dhaval\miniconda3\envs\cancer\Lib\contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.8750 - loss: 0.3143 - val_accuracy: 1.0000 - val_loss: 0.0944
Epoch 3/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 49s 3s/step - accuracy: 0.9296 - loss: 0.2369 - val_accuracy: 0.9688 - val_loss: 0.1369
Epoch 4/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.8125 - loss: 0.5170 - val_accuracy: 1.0000 - val_loss: 0.0037
Epoch 5/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 51s 3s/step - accuracy: 0.9356 - loss: 0.1764 - val_accuracy: 1.0000 - val_loss: 0.0776
